# 05 — Emotion vectors: geometry + how emotion loads into stories

**Project:** [EmoVecLLM](https://github.com/drgzkr/EmoVecLLM) — replicating Anthropic 2026 emotion vectors on open-source LLMs.

The figures notebook for **nb04**. It reads nb04's three artefacts and draws preliminary figures — no model, no GPU, pure `numpy`/`matplotlib`/`sklearn`, so it runs on whatever is already on disk (including a partial **demo** run).

**Figures**

1. **Availability** — segments per emotion + neutral baseline coverage (the health check on a partial run).
2. **Geometry** — emotion vectors embedded in 2D by **PCA and UMAP/t-SNE** side by side, coloured by k-means cluster; plus the emotion×emotion similarity matrix under **cosine and correlation** for comparison.
3. **Loading curves** — the headline: each emotion's average vector **projected onto the word-by-word cumulative feature timeseries** of its stories, showing the emotion direction being *loaded* as the model reads. We contrast the **matched** emotion vector against **mismatched** ones to show specificity. The first-token attention sink is detected empirically and removed.

> Everything is keyed off whatever `features/{spec_hash}/{target_model}/` directory nb04 last wrote (auto-discovered, or set `EMOVEC_FEATURES_DIR`).

## 0 — Setup

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn", "matplotlib", "umap-learn"], check=False)


In [ ]:
import json, os, re
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

def _env(name, default):
    v = os.environ.get(name)
    return v if v not in (None, "") else default

def _env_int(name, default):
    v = os.environ.get(name)
    return int(v) if v not in (None, "") else default

def _env_flag(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "on")

def _unit(v, axis=-1, eps=1e-12):
    return v / (np.linalg.norm(v, axis=axis, keepdims=True) + eps)

MOUNT_DRIVE = _env_flag("EMOVEC_MOUNT_DRIVE", IN_COLAB)
if os.environ.get("EMOVEC_WORK_DIR"):
    WORK_DIR = Path(os.environ["EMOVEC_WORK_DIR"]).expanduser()
elif IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/EmoVecLLM")
elif IN_COLAB:
    WORK_DIR = Path("/content/EmoVecLLM")
else:
    WORK_DIR = Path("..").resolve()

DATA_DIR = WORK_DIR / "data" / "processed"
FIG_DIR = WORK_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 10, "axes.labelsize": 9,
})
print(f"WORK_DIR = {WORK_DIR}")


## 1 — Locate nb04's features

Auto-discovers the `features/{spec_hash}/{target_model}/` with the **most segments** (so a real-model run, e.g. Qwen-7B with 3691 segments, always outranks a gpt2 demo) — all candidates are listed so the pick is visible. Override with `EMOVEC_FEATURES_DIR` (exact dir) or `EMOVEC_TARGET_MODEL` (substring filter, e.g. `Qwen`).

In [ ]:
FEATURES_DIR = _env("EMOVEC_FEATURES_DIR", "")
PREFER_MODEL = _env("EMOVEC_TARGET_MODEL", "")   # substring filter, e.g. "Qwen"
if FEATURES_DIR:
    feat_dir = Path(FEATURES_DIR).expanduser()
else:
    manifests = sorted((DATA_DIR / "features").rglob("features_manifest.json"))
    assert manifests, (f"no features under {DATA_DIR/'features'} — run nb04 first, "
                       f"or set EMOVEC_FEATURES_DIR")
    def _info(p):
        try:
            m = json.loads(p.read_text())
            return m.get("target_model", p.parent.name), m.get("n_segments", 0)
        except Exception:
            return p.parent.name, 0
    for p in manifests:
        tm, n = _info(p)
        print(f"candidate: {tm:<36s} n_segments={n:<6d} {p.parent}")
    pool = [p for p in manifests if PREFER_MODEL in _info(p)[0]] if PREFER_MODEL else manifests
    assert pool, f"no features match EMOVEC_TARGET_MODEL={PREFER_MODEL!r}"
    feat_dir = max(pool, key=lambda p: _info(p)[1]).parent
manifest = json.loads((feat_dir / "features_manifest.json").read_text())
print(f"features dir : {feat_dir}")
print(json.dumps(manifest, indent=2))


In [ ]:
seg = np.load(feat_dir / "segment_features.npz", allow_pickle=True)
vec = np.load(feat_dir / "emotion_vectors.npz", allow_pickle=True)
cum = np.load(feat_dir / "cumulative_timeseries.npz", allow_pickle=True)

X            = seg["X"]                       # (n_seg, n_layers, d)
seg_emotions = seg["emotions"]
seg_kinds    = seg["kinds"]

V            = vec["vectors"]                 # (E, n_layers, d)
emotions     = list(vec["emotions"])
cluster_lab  = vec["cluster_labels"]
CLUSTER_L    = int(vec["cluster_layer"])
CUM_L        = int(vec["cum_layer"])
baseline     = vec["baseline_mean"]          # (n_layers, d)
baseline_mode = str(vec["baseline_mode"])
n_per_emotion = vec["n_per_emotion"]
TARGET_MODEL = str(vec["target_model"])
SKIP_FIRST_DONE = bool(vec["skip_first"]) if "skip_first" in vec.files else False  # sink dropped in nb04?

cum_curves   = cum["cum"]                     # object array of (seq, n_sel_layers, d)
cum_emotions = cum["cum_emotions"]
cum_kinds    = cum["cum_kinds"]
cum_tokens   = cum["cum_tokens"]
cum_layer_idx  = [int(x) for x in cum["cum_layer_indices"]]
cum_proj_layer = int(cum["cum_proj_layer"])
PROJ_POS       = cum_layer_idx.index(cum_proj_layer)   # which slice of axis-1 to project

E, n_layers, d_model = V.shape
DEMO = manifest.get("demo", False)
print(f"target={TARGET_MODEL}  emotions={E}  layers={n_layers}  d={d_model}")
print(f"baseline={baseline_mode}  cluster_layer={CLUSTER_L}  proj_layer={CUM_L}  "
      f"sink-free(nb04 skip_first)={SKIP_FIRST_DONE}")
print(f"per-step timeseries: {len(cum_curves)} stories, stored layers={cum_layer_idx} "
      f"(project at layer {cum_proj_layer})"
      + ("   [DEMO run — figures are illustrative]" if DEMO else ""))


## 2 — Figure 1 · What's available

Segments per emotion (sorted), with the neutral-baseline count called out. On a partial run this immediately shows which emotions are under-sampled and whether a baseline exists at all.

In [ ]:
order = np.argsort(n_per_emotion)[::-1]
emos_sorted = [emotions[i] for i in order]
counts_sorted = n_per_emotion[order]
n_neutral = int(np.sum(np.isin(seg_kinds, ["neutral_dialogue", "neutral_story"])))

fig, ax = plt.subplots(figsize=(max(6, 0.16 * E + 2), 3.4))
ax.bar(range(E), counts_sorted, color="#3b6ea5", width=0.9)
ax.axhline(np.mean(counts_sorted), color="#d1495b", lw=1, ls="--",
           label=f"mean {np.mean(counts_sorted):.1f}")
ax.set_xlim(-0.5, E - 0.5)
ax.set_xlabel(f"emotion (n={E}, sorted by coverage)")
ax.set_ylabel("segments")
ax.set_title(f"Story coverage per emotion · target={TARGET_MODEL} · "
             f"neutral baseline: {n_neutral} segs ({baseline_mode})")
# Label only the extremes to avoid clutter.
show = list(range(min(6, E))) + list(range(max(0, E - 6), E))
ax.set_xticks(show)
ax.set_xticklabels([emos_sorted[i] for i in show], rotation=45, ha="right", fontsize=6)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "nb05_fig1_coverage.png", bbox_inches="tight")
plt.show()
print(f"total segments {int(np.sum(n_per_emotion))} across {E} emotions; "
      f"neutral {n_neutral}")


## 3 — Figure 2 · Emotion-vector geometry

The emotion vectors at layer `CLUSTER_L` embedded in 2D two ways, coloured by k-means cluster: **PCA** (left — linear, global structure; nb07 will align PCs to Warriner valence/arousal) and **UMAP** (right — nonlinear, local neighbourhoods; falls back to **t-SNE** when `umap-learn` isn't installed). PCA answers "what are the big axes?", the nonlinear map answers "which emotions sit together?" — read them jointly.

In [ ]:
from sklearn.decomposition import PCA

Vc = V[:, CLUSTER_L, :]                       # (E, d)
Vu = _unit(Vc)
proj = PCA(n_components=2).fit(Vu)
P2 = proj.transform(Vu)
evr = proj.explained_variance_ratio_
# Deterministic PC sign: make each PC's largest-magnitude loading positive, so
# reruns on the same vectors orient the same way (PCA signs are otherwise
# arbitrary, which makes the scatter look like it "changed" when it hasn't).
for _i in range(P2.shape[1]):
    _j = np.argmax(np.abs(proj.components_[_i]))
    if proj.components_[_i, _j] < 0:
        P2[:, _i] *= -1

# Nonlinear companion view: UMAP if available, else t-SNE (both on the unit
# vectors with cosine distance, so they see the same geometry as the matrices).
try:
    from umap import UMAP
    N2 = UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
              metric="cosine", random_state=0).fit_transform(Vu)
    EMB_NAME = "UMAP"
except Exception:
    from sklearn.manifold import TSNE
    N2 = TSNE(n_components=2, perplexity=min(30, E - 1), metric="cosine",
              init="pca", random_state=0).fit_transform(Vu)
    EMB_NAME = "t-SNE"

k = int(cluster_lab.max()) + 1
cmap = plt.cm.tab10 if k <= 10 else plt.cm.tab20

def _emb_scatter(ax, pts, title, xlab, ylab):
    for c in range(k):
        m = cluster_lab == c
        ax.scatter(pts[m, 0], pts[m, 1], s=42, color=cmap(c % cmap.N),
                   edgecolor="white", linewidth=0.5, label=f"c{c}", zorder=2)
    lab_idx = range(E) if E <= 40 else np.argsort(np.linalg.norm(pts - pts.mean(0), axis=1))[::-1][:40]
    for i in lab_idx:
        ax.annotate(emotions[i], (pts[i, 0], pts[i, 1]), fontsize=6, alpha=0.8,
                    xytext=(3, 2), textcoords="offset points")
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.set_title(title)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.6))
_emb_scatter(axL, P2, f"PCA (layer {CLUSTER_L})",
             f"PC1 ({evr[0]*100:.1f}%)", f"PC2 ({evr[1]*100:.1f}%)")
_emb_scatter(axR, N2, f"{EMB_NAME} (cosine, layer {CLUSTER_L})",
             f"{EMB_NAME}-1", f"{EMB_NAME}-2")
axL.legend(frameon=False, fontsize=7, ncol=2, loc="best")
fig.suptitle(f"Emotion-vector geometry — coloured by k-means cluster · target={TARGET_MODEL}", y=1.0)
fig.tight_layout()
fig.savefig(FIG_DIR / "nb05_fig2_geometry.png", bbox_inches="tight")
plt.show()


### 3b — Cosine vs correlation similarity

The same emotion×emotion similarity matrix under two metrics, rows/cols ordered by cluster (diagonal blocks = coherent emotion families): **cosine** on the raw vectors, and **Pearson correlation** — which is exactly cosine after centring each vector across its dimensions, so it discounts any shared offset common to all dimensions. If the two matrices look alike, the vectors carry no such offset and the metric choice is immaterial; if correlation shows crisper blocks, prefer it (the printed off-diagonal summary quantifies the difference).

In [ ]:
ord_by_cluster = np.argsort(cluster_lab, kind="stable")
S_cos = _unit(Vc) @ _unit(Vc).T               # cosine on raw vectors
S_cor = np.corrcoef(Vc)                       # = cosine after per-vector centring

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.6))
bounds = np.where(np.diff(cluster_lab[ord_by_cluster]) != 0)[0] + 0.5
for ax, S, name in [(axes[0], S_cos, "cosine"), (axes[1], S_cor, "correlation")]:
    So = S[np.ix_(ord_by_cluster, ord_by_cluster)]
    vmax = float(np.percentile(np.abs(So - np.eye(E)), 99)) or 1.0
    im = ax.imshow(So, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    for b in bounds:
        ax.axhline(b, color="k", lw=0.4); ax.axvline(b, color="k", lw=0.4)
    ax.set_title(f"{name} (ordered by cluster)")
    if E <= 40:
        ax.set_xticks(range(E)); ax.set_yticks(range(E))
        ax.set_xticklabels([emotions[i] for i in ord_by_cluster], rotation=90, fontsize=5)
        ax.set_yticklabels([emotions[i] for i in ord_by_cluster], fontsize=5)
    fig.colorbar(im, ax=ax, shrink=0.75, label=name)
fig.tight_layout()
fig.savefig(FIG_DIR / "nb05_fig2b_similarity.png", bbox_inches="tight")
plt.show()

off = ~np.eye(E, dtype=bool)
same = (cluster_lab[:, None] == cluster_lab[None, :]) & off
diff = (cluster_lab[:, None] != cluster_lab[None, :])
for S, name in [(S_cos, "cosine"), (S_cor, "correlation")]:
    print(f"{name:>12s}: within-cluster {S[same].mean():+.3f}  between {S[diff].mean():+.3f}  "
          f"separation {S[same].mean() - S[diff].mean():+.3f}")
print(f"cosine-vs-correlation agreement: r={np.corrcoef(S_cos[off], S_cor[off])[0,1]:.4f}")


## 4 — Figure 3 · How emotion is *loaded* into a story

The per-step timeseries `f[t]` is the residual after the model has read the prefix `tokens[:t+1]` — its state at each point in the story (the cumulation is done by the model's attention). Project it (centred on the baseline, at the projection layer) onto the **unit emotion vector** to get a scalar per token:

$$\text{load}_e(t) = \big(\,f(t) - \text{baseline}\,\big)\cdot \hat{v}_e$$

For a grid of example stories we plot the load along their *own* emotion (**matched**) against the mean of the *other* emotions' vectors (**mismatched**). A matched curve that climbs above mismatched as the story unfolds is the signature of the emotion direction being progressively written into the residual stream.

**Picking examples is a first-class knob**: set `PICK_EMOTIONS` (names) or `PICK_STORIES` (timeseries indices, shown in each panel title) in the cell below — or the env equivalents `EMOVEC_EXAMPLE_EMOTIONS` / `EMOVEC_EXAMPLE_STORIES` / `EMOVEC_N_EXAMPLES`. By default 12 emotions are spread evenly across the available alphabet.

> **Sink handling is measured, not assumed.** nb04's `skip_first` removes the attention-sink token from the *pooled* features and the baseline — but the stored per-step timeseries keeps the raw token-0 state, which on real models is sink-dominated (Qwen-7B: token-0 norm ≈ 17× the rest, and it saturates fp16). So this cell measures the token-0/rest norm ratio directly on the loaded data and drops the first token(s) whenever the sink is present, regardless of the flag. Centring: when the *baseline* is sink-free (`skip_first=True`) the raw absolute loading is meaningful; for legacy data with a contaminated baseline we fall back to **emotion-specific centring** (each curve minus the per-token mean across emotions). Override with `EMOVEC_DROP_FIRST` / `EMOVEC_CENTER`. Background: `meta/journal/2026-06-04_investigation_first_token_loading.md`.

In [ ]:
emo_index = {e: i for i, e in enumerate(emotions)}
vhat = _unit(V[:, CUM_L, :])                  # (E, d) unit emotion dirs at cum layer
base_c = baseline[CUM_L]                       # (d,)

# Detect the token-0 sink empirically in the stored timeseries (don't trust
# skip_first: it governs pooling/baseline, not the per-step traces).
_r0, _rr = [], []
for _j in range(len(cum_curves)):
    if cum_kinds[_j] != "emotion_story" or cum_curves[_j].shape[0] < 3:
        continue
    _n = np.linalg.norm(np.asarray(cum_curves[_j][:, PROJ_POS, :], dtype=np.float64), axis=1)
    _r0.append(_n[0]); _rr.append(_n[1:].mean())
SINK_RATIO = float(np.mean(_r0) / max(np.mean(_rr), 1e-9)) if _r0 else 1.0
SINK_PRESENT = SINK_RATIO > 3.0
DROP_FIRST_TOKENS = _env_int("EMOVEC_DROP_FIRST", 1 if SINK_PRESENT else 0)
CENTER_INDIV = _env("EMOVEC_CENTER", "none" if SKIP_FIRST_DONE else "cross_emotion")
print(f"token-0 sink: norm ratio {SINK_RATIO:.1f}x -> "
      + (f"PRESENT — dropping first {DROP_FIRST_TOKENS} token(s) from all loading plots"
         if DROP_FIRST_TOKENS else "absent — keeping all tokens")
      + f"   [centring={CENTER_INDIV}; override EMOVEC_DROP_FIRST / EMOVEC_CENTER]")

def load_all(feat_seq):
    'feat_seq (seq, n_sel_layers, d) -> (E, seq): loadings on ALL emotions, one matmul.'
    f = feat_seq[:, PROJ_POS, :].astype(np.float64)        # (seq, d) at projection layer
    return ((f - base_c) @ vhat.T).T

def load_curve(feat_seq, e_idx):
    'feat_seq (seq, n_sel_layers, d): project the projection-layer residual onto emotion e_idx.'
    f = feat_seq[:, PROJ_POS, :].astype(np.float64)        # (seq, d) at projection layer
    return (f - base_c) @ vhat[e_idx]

def matched_mismatched(seqd, ei, d0):
    'Return (matched, mismatched) loading curves from token d0 on; emotion-centred if set.'
    allc = load_all(seqd)[:, d0:]                                      # (E, T)
    mism = (allc.sum(0) - allc[ei]) / (E - 1)                          # mean over others
    ref = allc.mean(0) if CENTER_INDIV == "cross_emotion" else 0.0     # per-token mean over emotions
    return allc[ei] - ref, mism - ref

# ── Picking what to plot ─────────────────────────────────────────────────────
# story_by_emotion maps emotion -> indices into the stored timeseries. Choose via
#   PICK_STORIES   exact timeseries indices (or env EMOVEC_EXAMPLE_STORIES="0,17,42") — wins
#   PICK_EMOTIONS  one story per named emotion (or env EMOVEC_EXAMPLE_EMOTIONS="serene,outraged")
#   N_EXAMPLES     otherwise: this many emotions spread evenly A→Z (env EMOVEC_N_EXAMPLES)
story_by_emotion = {}
for j in range(len(cum_curves)):
    if cum_kinds[j] == "emotion_story" and cum_emotions[j] in emo_index:
        story_by_emotion.setdefault(str(cum_emotions[j]), []).append(j)
print(f"{sum(len(v) for v in story_by_emotion.values())} stored story timeseries across "
      f"{len(story_by_emotion)} emotions — e.g. "
      + ", ".join(f"{e}:{v}" for e, v in list(story_by_emotion.items())[:4]) + ", …")

PICK_STORIES  = [int(s) for s in _env("EMOVEC_EXAMPLE_STORIES", "").split(",") if s.strip()]
PICK_EMOTIONS = [s.strip() for s in _env("EMOVEC_EXAMPLE_EMOTIONS", "").split(",") if s.strip()]
N_EXAMPLES    = _env_int("EMOVEC_N_EXAMPLES", 12)

if PICK_STORIES:
    ex = [j for j in PICK_STORIES if 0 <= j < len(cum_curves)]
elif PICK_EMOTIONS:
    missing = [e for e in PICK_EMOTIONS if e not in story_by_emotion]
    if missing:
        print(f"no stored timeseries for: {missing} (available: {sorted(story_by_emotion)[:10]}…)")
    ex = [story_by_emotion[e][0] for e in PICK_EMOTIONS if e in story_by_emotion]
else:
    avail = sorted(story_by_emotion)
    sel = [avail[i] for i in np.linspace(0, len(avail) - 1,
                                         min(N_EXAMPLES, len(avail))).astype(int)]
    ex = [story_by_emotion[e][0] for e in sel]

if ex:
    ncol = 4 if len(ex) > 9 else min(3, len(ex))
    nrow = int(np.ceil(len(ex) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.1 * nrow),
                             squeeze=False)
    for ax in axes.flat:
        ax.set_visible(False)
    ylab = ("loading − mean over emotions" if CENTER_INDIV == "cross_emotion"
            else "projection")
    for ax, j in zip(axes.flat, ex):
        ax.set_visible(True)
        e = cum_emotions[j]; ei = emo_index[e]
        seqd = cum_curves[j]
        d0 = min(DROP_FIRST_TOKENS, seqd.shape[0] - 1)
        matched, mismatched = matched_mismatched(seqd, ei, d0)
        t = np.arange(d0, d0 + len(matched))
        ax.plot(t, matched, color="#d1495b", lw=1.8, label=f"matched: {e}")
        ax.plot(t, mismatched, color="#8d99ae", lw=1.2, ls="--", label="mean other")
        ax.axhline(0, color="k", lw=0.5, alpha=0.4)
        ax.set_title(f"{e} · story #{j}", fontsize=9)
        ax.set_xlabel(f"token position (first {d0} dropped)")
        ax.set_ylabel(ylab)
        ax.legend(frameon=False, fontsize=7)
    fig.suptitle(f"Emotion loading along the story (layer {CUM_L}, target={TARGET_MODEL}"
                 f", centre={CENTER_INDIV})", y=1.01)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "nb05_fig3_loading_examples.png", bbox_inches="tight")
    plt.show()
else:
    print("no emotion-story cumulative timeseries found — re-run nb04 with stories present.")


### 4b — Aggregate loading: matched vs mismatched

Each story's load curve is resampled onto a common fractional-position axis (0 → start, 1 → end) and averaged — over **every** stored timeseries (nb04 caps how many per-step traces it saves: `n_timeseries` in the manifest, here printed below; the pooled features in §2-3 still use all segments). Sink tokens are dropped per §4. If the emotion direction is genuinely written during reading, the **matched** mean rises above the **mismatched** mean and the gap widens toward the end.

In [ ]:
def resample(y, n=50):
    y = np.asarray(y, dtype=np.float64)
    if len(y) < 2:
        return np.full(n, y[0] if len(y) else 0.0)
    xp = np.linspace(0, 1, len(y))
    return np.interp(np.linspace(0, 1, n), xp, y)

N = 50
matched_all, mismatched_all = [], []
for j in range(len(cum_curves)):
    e = cum_emotions[j]
    if cum_kinds[j] != "emotion_story" or e not in emo_index:
        continue
    ei = emo_index[e]; seqd = cum_curves[j]
    d0 = min(DROP_FIRST_TOKENS, seqd.shape[0] - 1)
    allc = load_all(seqd)[:, d0:]                          # (E, T) one matmul per story
    matched_all.append(resample(allc[ei], N))
    mismatched_all.append(resample((allc.sum(0) - allc[ei]) / (E - 1), N))

if matched_all:
    M = np.stack(matched_all); MM = np.stack(mismatched_all)
    print(f"aggregating ALL {len(matched_all)} stored story timeseries "
          f"(of {int(manifest.get('n_timeseries', len(cum_curves)))} saved by nb04; "
          f"first {DROP_FIRST_TOKENS} token(s) dropped per curve)")
    x = np.linspace(0, 1, N)
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for arr, col, lab in [(M, "#d1495b", "matched emotion"),
                          (MM, "#8d99ae", "mismatched (mean other)")]:
        mu = arr.mean(0); se = arr.std(0) / np.sqrt(len(arr))
        ax.plot(x, mu, color=col, lw=2, label=lab)
        ax.fill_between(x, mu - se, mu + se, color=col, alpha=0.2)
    ax.axhline(0, color="k", lw=0.5, alpha=0.4)
    ax.set_xlabel("fractional story position (0=start, 1=end)")
    ax.set_ylabel("projection onto emotion vector")
    ax.set_title(f"Mean emotion loading ({len(matched_all)} stories, layer {CUM_L})")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "nb05_fig3b_loading_aggregate.png", bbox_inches="tight")
    plt.show()
    gap = (M[:, -1] - MM[:, -1])
    print(f"end-of-story matched − mismatched: mean {gap.mean():+.3f} "
          f"(matched > mismatched in {100*np.mean(gap>0):.0f}% of stories)")
else:
    print("no aggregate curves available yet.")


### 4c — Diagnostic: the first-token attention sink

Quantifies the first-token attention sink — or **confirms it's gone** when nb04 ran with `skip_first` (then the norm ratio should be ≈1× and there's no token-0 spike). For every emotion story we compare the residual-stream **norm** at token 0 vs the rest, and the **loading** at token 0 vs the rest, then plot mean norm & loading over the first ~25 positions. A large norm ratio (≈10–100×) with a huge token-0 loading is the attention-sink / massive-activation signature (full write-up in `meta/journal/2026-06-04_investigation_first_token_loading.md`).

In [ ]:
rows = []
for j in range(len(cum_curves)):
    e = cum_emotions[j]
    if cum_kinds[j] != "emotion_story" or e not in emo_index:
        continue
    f = np.asarray(cum_curves[j][:, PROJ_POS, :], dtype=np.float64)   # (seq, d)
    if f.shape[0] < 2:
        continue
    nrm = np.linalg.norm(f, axis=1)
    lc = load_curve(cum_curves[j], emo_index[e])
    rows.append((nrm[0], nrm[1:].mean(), lc[0], lc[1:].mean()))

if rows:
    A = np.array(rows)
    n0, nr, l0, lr = A[:, 0].mean(), A[:, 1].mean(), A[:, 2].mean(), A[:, 3].mean()
    print(f"residual norm   token0={n0:10.1f}   rest={nr:10.1f}   ratio={n0/max(nr,1e-9):7.1f}x")
    print(f"matched loading token0={l0:+10.1f}   rest={lr:+10.1f}")

    def _mean_over_stories(fn, P):
        out = np.full(P, np.nan)
        for p in range(P):
            vals = [fn(j, p) for j in range(len(cum_curves))
                    if cum_kinds[j] == "emotion_story" and cum_emotions[j] in emo_index
                    and cum_curves[j].shape[0] > p]
            if vals:
                out[p] = np.mean(vals)
        return out

    P = 25
    norm_pos = _mean_over_stories(   # float64: token-0 norms overflow fp16
        lambda j, p: np.linalg.norm(np.asarray(cum_curves[j][p, PROJ_POS, :], dtype=np.float64)), P)
    load_pos = _mean_over_stories(
        lambda j, p: load_curve(cum_curves[j], emo_index[cum_emotions[j]])[p], P)

    fig, (a0, a1) = plt.subplots(1, 2, figsize=(10, 3.4))
    a0.plot(range(P), norm_pos, marker="o", ms=3, color="#3b6ea5")
    a0.set_yscale("log"); a0.set_xlabel("token position"); a0.set_ylabel("‖residual‖ (log)")
    a0.set_title(f"Residual norm by position (sink at 0), layer {CUM_L}")
    a1.plot(range(P), load_pos, marker="o", ms=3, color="#d1495b")
    a1.axhline(0, color="k", lw=0.5, alpha=0.4)
    a1.set_xlabel("token position"); a1.set_ylabel("matched loading")
    a1.set_title("Matched loading by position")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "nb05_fig4d_sink.png", bbox_inches="tight")
    plt.show()
else:
    print("no emotion-story timeseries to diagnose.")


### 4d — The raw token × feature timeseries + temporal correlation (per layer)

This is the object the loading curve is computed *from*: nb04's per-step array `cum[j]` has shape `(seq, n_sel_layers, d_model)`. For one story (`EXAMPLE`) and a set of stored layers (`LAYERS_TO_SHOW`), each row shows two panels:

- **left** — the **feature × token** heatmap (z-scored per feature across tokens), `token position on x` (left→right = the model cumulatively reading the story), feature on y;
- **right** — the **token × token** correlation matrix (`np.corrcoef` across features), i.e. how similar the model's state is between any two reading positions. Block structure = stretches of the story the model represents alike; an off-diagonal that fades shows the state drifting as the narrative develops.

`LAYERS_TO_SHOW` is intersected with the stored `cum_layer_idx`. For large `d_model` set `TOP_FEATURES` to show only the highest-variance features in the left panel (the correlation still uses all features).

In [ ]:
LAYERS_TO_SHOW = cum_layer_idx[:4]    # any subset of the stored layers
EXAMPLE        = 0                     # index into the stored per-step timeseries
TOP_FEATURES   = 0                     # 0 = all features (left panel only)

def _feat_by_token(feat):
    'feat (seq, d) -> (d, seq) z-scored per feature across tokens.'
    f = feat.astype(np.float64)
    return ((f - f.mean(0)) / (f.std(0) + 1e-12)).T

layers = [L for L in LAYERS_TO_SHOW if L in cum_layer_idx]
if not len(cum_curves):
    print("no per-step timeseries saved — run nb04 with stories present.")
elif not layers:
    print(f"none of {LAYERS_TO_SHOW} are stored; available: {cum_layer_idx}. "
          f"Re-run nb04 with EMOVEC_CUM_LAYERS including them (e.g. 'all').")
else:
    toks = [str(t).replace("Ġ", " ").replace("Ċ", "/n").strip()
            for t in list(cum_tokens[EXAMPLE])]
    fig, axs = plt.subplots(len(layers), 2, squeeze=False,
                            figsize=(11, 2.7 * len(layers)),
                            gridspec_kw={"width_ratios": [3, 1]})
    fig.suptitle(f"Story #{EXAMPLE} — {cum_emotions[EXAMPLE]} · target={TARGET_MODEL}",
                 y=1.0)
    for r, L in enumerate(layers):
        lp = cum_layer_idx.index(L)
        ft = _feat_by_token(cum_curves[EXAMPLE][:, lp, :])      # (d, seq)
        seq = ft.shape[1]
        ft_show = ft
        if TOP_FEATURES and TOP_FEATURES < ft.shape[0]:
            keep = np.argsort(ft.var(1))[::-1][:TOP_FEATURES]
            ft_show = ft[keep]
        ax0, ax1 = axs[r, 0], axs[r, 1]
        vmax = float(np.percentile(np.abs(ft_show), 99)) or 1.0
        ax0.imshow(ft_show, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   interpolation="nearest")
        ax0.set_ylabel(f"layer {L}\nfeature")
        if r == 0:
            ax0.set_title("feature x token  (z-scored)", fontsize=9, loc="left")
        # token x token correlation across features.
        cm = np.corrcoef(ft.T)                                   # (seq, seq)
        ax1.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
        if r == 0:
            ax1.set_title("token x token corr", fontsize=9, loc="left")
        if r == len(layers) - 1 and seq <= 60:
            ax0.set_xticks(range(seq)); ax0.set_xticklabels(toks[:seq], rotation=90, fontsize=6)
        else:
            ax0.set_xticks([])
        ax1.set_xticks([]); ax1.set_yticks([])
    fig.tight_layout()
    fig.savefig(FIG_DIR / "nb05_fig4_token_feature.png", bbox_inches="tight")
    plt.show()
    print(f"plotted story {EXAMPLE} ({cum_emotions[EXAMPLE]}) at layers {layers}")


## 5 — Reading these figures + the neutral-baseline question

**Demo caveat.** With `EMOVEC_DEMO=1` and a tiny target (`gpt2`), expect noisy geometry and weak loading separation — the *plumbing* is what's being validated, not the science. Scale up via `EMOVEC_DEMO=0`, a real target (`EleutherAI/pythia-1.4b` → the headline 7–8B models), and full story coverage.

**The neutral baseline shapes every panel here.** The loading curves are projections onto baseline-centred vectors, so what counts as "neutral" sets the zero. The active baseline mode is printed in §1 (`baseline_mode`). Two regimes (see nb04 §6 for the switch):

- **Neutral dialogue** (Anthropic): form-mismatched to our story prose, so a story-vs-dialogue *style* axis contaminates the vectors → prefer `EMOVEC_BASELINE=project_pcs`, and read the loading curves knowing some of the matched rise is shared "narrative" style, not emotion.
- **Neutral stories** (our likely choice): style-matched, so `neutral_mean` should clean the space with less PC surgery, *but* watch that we don't subtract genuine shared narrative content. The matched-vs-mismatched gap in §4b is the quickest read on whether the baseline is doing the right thing.

**Next:** nb06 (held-out scoring on EmoBank/GoEmotions) and nb07 (PCA → Warriner valence/arousal alignment, k-means labelling). Both consume the same `features/` artefacts written by nb04.

In [ ]:
print("figures written to:", FIG_DIR)
for p in sorted(FIG_DIR.glob("nb05_*.png")):
    print("  ", p.name)
